# **PSA Intelligence Platform - Data Collection Pipeline**

## **SECTION 1: SOURCE REGISTRY**

List of government sites to pull from. Each source needs its own CSS selectors since every gov site's HTML template is different.

In [4]:
!pip install requests beautifulsoup4 pandas langdetect --quiet

import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time
import re
from datetime import datetime
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [5]:
SOURCE_REGISTRY = [
    {
        "name": "Kenya News Agency",
        "base_url": "https://www.kenyanews.go.ke",
        "listing_url": "https://www.kenyanews.go.ke/news/",
        "listing_page_pattern": "page/{page}/",
        "link_selector": "article a[href], h2 a[href], h3 a[href]",
        "title_selector": "h1",
        "body_selector": "div.entry-content, div.post-content, article",
        "date_selector": "time",
    },
]
HEADERS = {"User-Agent": "Mozilla/5.0 (research/education scraper - USIU DSA4020 project)"}
REQUEST_DELAY = 2
MAX_PAGES = 30
MIN_TEXT_LENGTH = 40

DOMAIN_KEYWORDS = {
    "Health": ["health", "vaccin", "disease", "hospital", "clinic", "malaria", "hiv", "cholera", "nutrition"],
    "Agriculture": ["farm", "crop", "livestock", "agricult", "irrigation", "fertiliz", "drought"],
    "Security": ["security", "police", "safety", "crime", "accident", "disaster", "flood", "fire"],
    "Education": ["school", "education", "student", "bursary", "scholarship", "literacy"],
    "Governance": ["county", "governor", "corruption", "devolution", "voter", "election", "government"],
}

PSA_TAXONOMY = {
    "Health": {
        "Disease Prevention and Control": ["malaria", "hiv", "aids", "tuberculosis", "tb ", "vaccin", "outbreak", "screening", "disease"],
        "Maternal and Child Health": ["maternal", "pregnan", "infant", "child health", "antenatal", "immuniz"],
        "Public Health Campaigns": ["hygiene", "sanitation", "diabetes", "non-communicable", "health promotion"],
        "Mental Health Awareness": ["mental health", "stress", "suicide", "counsel"],
        "Healthcare Access": ["medical camp", "health insurance", "nhif", "sha ", "hospital", "clinic"],
    },
    "Agriculture": {
        "Crop Production": ["crop", "yield", "pest", "drought-resistant", "seed"],
        "Livestock Management": ["livestock", "cattle", "grazing", "animal vaccin"],
        "Agribusiness and Market Access": ["market access", "cooperative", "fair trade", "agribusiness"],
        "Sustainable Farming": ["organic farming", "soil conservation", "water management", "sustainable"],
        "Agricultural Training": ["training", "irrigation", "fertiliz", "farming techniques"],
    },
    "Education": {
        "Access to Education": ["enrollment", "enrolment", "literacy", "free primary", "free secondary"],
        "Vocational Training": ["vocational", "technical education", "skills training"],
        "Civic Education": ["civic education", "civic rights", "education policy"],
        "Educational Resources": ["scholarship", "bursary", "textbook", "online course"],
        "School Safety and Inclusion": ["bullying", "disability", "safe school", "inclusion"],
    },
    "Security & Safety": {
        "Public Safety Awareness": ["road safety", "fire prevention", "disaster preparedness", "flood", "drought response"],
        "Crime Prevention": ["crime", "community policing", "report a crime"],
        "National Security": ["counter-terrorism", "border security", "peace-building", "peacebuilding"],
        "Gender-Based Violence": ["domestic violence", "sexual assault", "gender-based violence", "gbv", "survivors"],
        "Cybersecurity": ["online scam", "data privacy", "cyber", "mpesa fraud", "bank alert", "internet usage"],
    },
    "Governance": {
        "Anti-Corruption Initiatives": ["corruption", "eacc", "transparency"],
        "Public Participation": ["public forum", "policy consultation", "public participation"],
        "Elections and Voter Education": ["voter", "election", "electoral rights"],
        "Public Service Delivery": ["passport", "e-citizen", "ecitizen", "national id", "government services"],
        "Devolution and Local Governance": ["devolution", "county government", "local governance"],
    },
}

PSA_SIGNAL_PHRASES = [
    "urged", "urges", "caution", "cautioned", "warn", "warned", "warns",
    "advised", "advises", "reminded", "reminds", "encourage", "encouraged",
    "should", "must", "avoid", "ensure", "please note", "public notice",
    "advisory", "deadline", "register", "apply by", "report to", "seek help",
    "call for", "calls for", "alert", "notice to", "attention",
]

BOILERPLATE_PATTERNS = [
    r"Follow Us.*", r"Follow us.*", r"Read on.*", r"Share this.*",
    r"Related [Aa]rticles.*", r"©.*(Kenya News Agency|All [Rr]ights [Rr]eserved).*",
    r"Subscribe to our newsletter.*",
]

## **SECTION 2: SCRAPER**
Visits listing pages for each source in the registry, opens each article, and downloads the raw title/body/date/url.

In [6]:
def get_article_links(source, max_pages=MAX_PAGES):
    links = set()
    for page in range(1, max_pages + 1):
        url = source["listing_url"] if page == 1 else source["listing_url"] + source["listing_page_pattern"].format(page=page)
        try:
            resp = requests.get(url, headers=HEADERS, timeout=15, verify=False)  # gov site has SSL cert issues
            if resp.status_code != 200:
                print(f"[{source['name']}] Stopped at page {page}: status {resp.status_code}")
                break
            soup = BeautifulSoup(resp.text, "html.parser")
            found = soup.select(source["link_selector"])
            page_links = {a["href"] for a in found if a.get("href", "").startswith(source["base_url"])}
            if not page_links:
                print(f"[{source['name']}] No links on page {page}, stopping.")
                break
            links.update(page_links)
            print(f"[{source['name']}] Page {page}: {len(page_links)} links (total: {len(links)})")
            time.sleep(REQUEST_DELAY)
        except requests.RequestException as e:
            print(f"[{source['name']}] Error on page {page}: {e}")
            break
    return list(links)

In [7]:
def scrape_article(url, source):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15, verify=False)  # gov site has SSL cert issues
        soup = BeautifulSoup(resp.text, "html.parser")
        title_tag = soup.select_one(source["title_selector"])
        title = title_tag.get_text(strip=True) if title_tag else ""
        body_tag = soup.select_one(source["body_selector"])
        paragraphs = body_tag.find_all("p") if body_tag else []
        body = " ".join(p.get_text(strip=True) for p in paragraphs)
        date_tag = soup.select_one(source["date_selector"])
        date = date_tag.get("datetime", "") if date_tag else ""
        return {
            "title": title,
            "raw_text": f"{title}. {body}".strip(),
            "date": date,
            "url": url,
            "source_name": source["name"],
        }
    except requests.RequestException as e:
        print(f"Failed to scrape {url}: {e}")
        return None

In [8]:
def run_scraper():
    all_raw_records = []
    for source in SOURCE_REGISTRY:
        links = get_article_links(source)
        print(f"\n[{source['name']}] Total links found: {len(links)}")
        for i, link in enumerate(links):
            data = scrape_article(link, source)
            if data and data["raw_text"]:
                all_raw_records.append(data)
            if i % 10 == 0:
                print(f"[{source['name']}] Scraped {i}/{len(links)}")
            time.sleep(REQUEST_DELAY)
    return all_raw_records

## **SECTION 3: CLEANING**
Removes menus/footers, fixes spacing, detects language.

In [9]:
def clean_text(raw_text):
    text = raw_text
    for pattern in BOILERPLATE_PATTERNS:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\xa0", " ").replace("\u200b", "")
    return text.strip()

def detect_language_safe(text):
    try:
        if len(text) < 20:
            return "unknown"
        return detect(text)
    except Exception:
        return "unknown"

# **SECTION 4: PSA CLASSIFIER**
- Asks: "Is this a real PSA or random news?"
- Rule-based for now - scores directive/action-oriented language

In [10]:
def classify_is_psa(text, min_signals=1):
    text_lower = text.lower()
    signal_hits = [phrase for phrase in PSA_SIGNAL_PHRASES if phrase in text_lower]
    psa_score = len(signal_hits)
    is_psa = psa_score >= min_signals
    return is_psa, psa_score, signal_hits

## **SECTION 5: METADATA ENRICHMENT**
Tags domain, urgency, audience, keywords

In [11]:
def classify_domain(text):
    """Two-level classification: returns (Category, Sub-Category).
    Falls back to ('General', 'Uncategorized') if nothing matches."""
    text_lower = text.lower()
    for category, subcats in PSA_TAXONOMY.items():
        for subcat, keywords in subcats.items():
            if any(kw in text_lower for kw in keywords):
                return category, subcat
    return "General", "Uncategorized"

def classify_urgency(text):
    urgent_words = ["urgent", "immediately", "deadline", "alert", "warning", "emergency", "outbreak"]
    return "High" if any(w in text.lower() for w in urgent_words) else "Normal"

def classify_audience(category):
    mapping = {
        "Health": "General public / patients",
        "Agriculture": "Farmers",
        "Security & Safety": "General public",
        "Education": "Students / parents",
        "Governance": "Citizens / voters",
        "General": "General public",
    }
    return mapping.get(category, "General public")

## **SECTION 6: VALIDATION**
Rejects empty/duplicate/too-short junk.

In [12]:
def validate_record(text, lang, seen_texts):
    if not text or len(text) < MIN_TEXT_LENGTH:
        return False, "too_short_or_empty"
    if text in seen_texts:
        return False, "duplicate"
    return True, "ok"

## **SECTION 7: DATABASE**
Saves good records as active PSAs.

In [13]:
DB_PATH = "psa_database.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS psas (
            psa_id TEXT PRIMARY KEY,
            domain TEXT,
            sub_category TEXT,
            english TEXT,
            kiswahili TEXT,
            kikuyu TEXT,
            source TEXT,
            date TEXT,
            urgency TEXT,
            audience TEXT,
            psa_score INTEGER,
            lang_detected TEXT,
            scraped_at TEXT
        )
    """)
    conn.commit()
    return conn

In [14]:
def save_record(conn, record):
    conn.execute("""
        INSERT OR IGNORE INTO psas
        (psa_id, domain, sub_category, english, kiswahili, kikuyu, source, date, urgency, audience, psa_score, lang_detected, scraped_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        record["PSA_ID"], record["Domain"], record["Sub_Category"], record["English"], record["Kiswahili"], record["Kikuyu"],
        record["Source"], record["Date"], record["Urgency"], record["Audience"], record["psa_score"],
        record["lang_detected"], datetime.now().isoformat()
    ))
    conn.commit()

In [15]:
raw_records = run_scraper()

conn = init_db()
accepted_rows = []
rejected_log = []
seen_texts = set()

for idx, r in enumerate(raw_records, start=1):
    cleaned = clean_text(r["raw_text"])
    lang = detect_language_safe(cleaned)

    ok, reason = validate_record(cleaned, lang, seen_texts)
    if not ok:
        rejected_log.append({"url": r["url"], "reason": reason})
        continue

    is_psa, psa_score, signals = classify_is_psa(cleaned)
    if not is_psa:
        rejected_log.append({"url": r["url"], "reason": "not_psa_like", "score": psa_score})
        continue

    category, subcategory = classify_domain(cleaned)
    urgency = classify_urgency(cleaned)
    audience = classify_audience(category)

    record = {
        "PSA_ID": f"GOV_{r['source_name'].replace(' ', '_').upper()}_{idx:04d}",
        "Domain": category,
        "Sub_Category": subcategory,
        "English": cleaned,
        "Kiswahili": "",
        "Kikuyu": "",
        "Source": r["url"],
        "Date": r["date"],
        "Urgency": urgency,
        "Audience": audience,
        "psa_score": psa_score,
        "lang_detected": lang,
    }
    seen_texts.add(cleaned)
    save_record(conn, record)
    accepted_rows.append(record)

conn.close()
print(f"\nAccepted (active PSAs, saved to database): {len(accepted_rows)}")
print(f"Rejected: {len(rejected_log)}")

[Kenya News Agency] Page 1: 40 links (total: 40)
[Kenya News Agency] Page 2: 40 links (total: 56)
[Kenya News Agency] Page 3: 40 links (total: 72)
[Kenya News Agency] Page 4: 40 links (total: 85)
[Kenya News Agency] Page 5: 39 links (total: 99)
[Kenya News Agency] Page 6: 40 links (total: 113)
[Kenya News Agency] Page 7: 40 links (total: 125)
[Kenya News Agency] Page 8: 43 links (total: 136)
[Kenya News Agency] Page 9: 42 links (total: 150)
[Kenya News Agency] Page 10: 40 links (total: 159)
[Kenya News Agency] Page 11: 41 links (total: 169)
[Kenya News Agency] Page 12: 41 links (total: 181)
[Kenya News Agency] Page 13: 41 links (total: 193)
[Kenya News Agency] Page 14: 40 links (total: 207)
[Kenya News Agency] Page 15: 41 links (total: 218)
[Kenya News Agency] Page 16: 41 links (total: 230)
[Kenya News Agency] Page 17: 40 links (total: 241)
[Kenya News Agency] Page 18: 40 links (total: 252)
[Kenya News Agency] Page 19: 39 links (total: 262)
[Kenya News Agency] Page 20: 40 links (total:

## **SECTION 8: EXPORT**
CSV for course / modeling.

In [16]:
psa_dataset = pd.DataFrame(accepted_rows)
psa_dataset.to_csv("psa_dataset_final.csv", index=False)
pd.DataFrame(rejected_log).to_csv("psa_rejected_log.csv", index=False)
print("Saved psa_dataset_final.csv and psa_rejected_log.csv")

Saved psa_dataset_final.csv and psa_rejected_log.csv
